# Week 7 deep-dive analysis: forecast baseline, customer behaviour, priority pack, lead-time robustness
**CadetX Heavy Supplier, Inventory & Warehouse Analytics: Phase 3, opening**

This notebook re-runs the four Week 7 work items from the same source, combines their validation checks, and cross-checks them against each other.

| Item | Question | Owner | Folder |
|---|---|---|---|
| W7-1 | How well does a plain level forecast do, and can anything beat it? | Abdibasid | `w71_forecast_baseline/` |
| W7-2 | How do customers differ in recency, frequency and value? | Salim | `w72_customer_behaviour/` |
| W7-3 | Which products, branches and stock pairs need attention first? | Chandrashekhar | `w73_priority_pack/` |
| W7-4 | Does the demand-based reorder trigger still beat the existing levels when lead times vary? | Ibrahim | `w74_lead_time_variation/` |

Stock results here rest on one assumption: **sales scale assumed (Q10 unresolved)**.

W7-4 runs 1,000 simulated lead-time paths and takes about two minutes. Each standalone notebook finds the corrected merged data itself (or uses `CADETX_MERGED`); W7-3 also needs the Week 6 stock outputs (`week-06/standalone_question_reports/stock_kpis_and_q11_rerun/stock_outputs`, or `CADETX_WEEK6_STOCK`).

In [1]:
import os, io, json, pathlib, contextlib, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
HERE=pathlib.Path.cwd()
def find_week07():
    for base in [HERE,*HERE.parents]:
        for p in (base, base/"week-07"):
            if (p/"standalone_question_reports").is_dir(): return p
    return None
W7=find_week07(); assert W7 is not None, "Run this notebook from inside the week-07 folder."
REPORTS=W7/"standalone_question_reports"
ITEMS=[("W7-1 Forecast baseline","w71_forecast_baseline","w71_forecast_baseline.ipynb","validation_report_week07_w71.csv"),
       ("W7-2 Customer behaviour","w72_customer_behaviour","w72_customer_behaviour.ipynb","validation_report_week07_w72.csv"),
       ("W7-3 Priority pack","w73_priority_pack","w73_priority_pack.ipynb","validation_report_week07_w73.csv"),
       ("W7-4 Lead-time robustness","w74_lead_time_variation","w74_lead_time_variation.ipynb","validation_report_week07_w74.csv")]
print("week-07 folder:",W7)

week-07 folder: /tmp/claude-0/-home-claude/6bcba5f3-dda2-517e-bd95-e2227d15c22a/scratchpad/w7/final2/week-07


## Part 1. Re-run every Week 7 notebook
Each notebook is run from its own folder in a fresh namespace; its own checks stop it if any fails.

In [2]:
def run_notebook(folder,nbname):
    nb=json.load(open(folder/nbname)); ns={"__name__":"__main__"}; old=os.getcwd(); os.chdir(folder)
    try:
        buf=io.StringIO()
        with contextlib.redirect_stdout(buf):
            for c in nb["cells"]:
                if c["cell_type"]=="code": exec(compile("".join(c["source"]),nbname,"exec"),ns)
    finally: os.chdir(old)
    return buf.getvalue()
LOGS={}
for label,folder,nbname,val in ITEMS:
    LOGS[label]=run_notebook(REPORTS/folder,nbname); n=pd.read_csv(REPORTS/folder/val)
    print(f"{label}: {len(n)} checks, {int((n.Status=='PASS').sum())} PASS")

W7-1 Forecast baseline: 27 checks, 27 PASS
W7-2 Customer behaviour: 22 checks, 22 PASS
W7-3 Priority pack: 26 checks, 26 PASS
W7-4 Lead-time robustness: 30 checks, 30 PASS


## Part 2. One combined validation report

In [3]:
parts=[]
for label,folder,nbname,val in ITEMS:
    d=pd.read_csv(REPORTS/folder/val); d.insert(0,"Question",label); d=d.drop(columns=["Check_ID"],errors="ignore"); parts.append(d)
VAL=pd.concat(parts,ignore_index=True); VAL.insert(1,"Check_ID",range(1,len(VAL)+1))
VAL.to_csv(W7/"validation_report_week07.csv",index=False)
print(VAL.groupby("Question").Status.value_counts().unstack(fill_value=0))
assert (VAL.Status=="PASS").all(), "A check failed"; print(f"{len(VAL)} checks, all PASS")

Status                     PASS
Question                       
W7-1 Forecast baseline       27
W7-2 Customer behaviour      22
W7-3 Priority pack           26
W7-4 Lead-time robustness    30
105 checks, all PASS


## Part 3. Cross-checks between the three items
The three notebooks were written separately. They must agree on the shared facts.

In [4]:
xchecks=[]
def xcheck(name,expected,actual,ok):
    xchecks.append(dict(Question="Cross-check",Check_Item=name,Expected=str(expected),Actual=str(actual),Status="PASS" if ok else "FAIL")); print(("PASS " if ok else "FAIL ")+name+" | expected "+str(expected)+" | actual "+str(actual))
o1=REPORTS/"w71_forecast_baseline"/"w71_outputs"; o2=REPORTS/"w72_customer_behaviour"/"w72_outputs"; o3=REPORTS/"w73_priority_pack"/"w73_outputs"; o4=REPORTS/"w74_lead_time_variation"/"w74_outputs"
cust=pd.read_csv(o2/"w72_customer_metrics.csv"); cls=pd.read_csv(o3/"w73_class_a_products.csv"); br=pd.read_csv(o3/"w73_branch_table.csv"); allp=pd.read_csv(o3/"w73_all_products_margin.csv")
s=pd.read_csv(find_merged:=None or next((p for base in [W7,*W7.parents] for p in [base/"week-02/reports/week_02/merged"] if (p/"sales_orders_full.csv").exists()))/"sales_orders_full.csv",low_memory=False,usecols=["so_id","order_status","line_total"])
dl=s[s.order_status=="Delivered"]
xcheck("customer revenue (W7-2) equals product revenue (W7-3)",int(round(dl.line_total.sum())),f"{int(round(cust.revenue.sum()))} / {int(round(allp.revenue.sum()))}",int(round(cust.revenue.sum()))==int(round(allp.revenue.sum()))==int(round(dl.line_total.sum())))
xcheck("customer orders (W7-2) equal branch orders (W7-3)",18033,f"{int(cust.orders.sum())} / {int(br.total_orders.sum())}",int(cust.orders.sum())==int(br.total_orders.sum())==18033)
f1=pd.read_csv(o1/"w71_series_results.csv"); b1=f1[f1.group=="branch"].set_index("series").mean_monthly_value
b3=br.set_index("branch_id").orders_per_month
xcheck("orders a month per branch: W7-1 series means equal W7-3 table (6 of 6)","equal",f"{int((np.isclose(b1.reindex(b3.index),b3)).sum())} of 6",bool(np.isclose(b1.reindex(b3.index),b3).all()))
t1=f1[(f1.group=="total")&(f1.series=="revenue")].mean_monthly_value.iloc[0]
xcheck("W7-1 mean monthly revenue times 72 equals total revenue",int(round(dl.line_total.sum())),int(round(t1*72)),abs(t1*72-dl.line_total.sum())<1)
fc=pd.read_csv(o1/"w71_forecast_2025.csv"); p1=f1[f1.group=="product"].set_index("series").mean_monthly_value
xcheck("product units (W7-1) equal class A units (W7-3) for the ten class A products","equal","equal" if np.allclose((p1.reindex(cls.product_id)*72).values,cls.units.values) else "differ",bool(np.allclose((p1.reindex(cls.product_id)*72).values,cls.units.values)))
pr4=pd.read_csv(o4/"w74_pair_results.csv"); a4=pr4[pr4.lead_time_case.str.startswith("constant")&pr4.policy.str.startswith("A")]
risk4=set(zip(a4[a4.fill_rate<0.95].product_id,a4[a4.fill_rate<0.95].branch_id)); risk3=pd.read_csv(o3/"w73_at_risk_stock_pairs.csv"); risk3=set(zip(risk3.product_id,risk3.branch_id))
xcheck("W7-4 constant-lead policy A fails 95% fill for the same 72 pairs as the W7-3 at-risk list",72,f"{len(risk4)} / {len(risk3)}; identical sets: {risk4==risk3}",len(risk4)==len(risk3)==72 and risk4==risk3)
xcheck("W7-4 covers the same 180 pairs as the W7-3 category table",180,f"{len(pr4[['product_id','branch_id']].drop_duplicates())} / {int(pd.read_csv(o3/'w73_at_risk_by_category.csv').pairs.sum())}",len(pr4[['product_id','branch_id']].drop_duplicates())==180==int(pd.read_csv(o3/'w73_at_risk_by_category.csv').pairs.sum()))
xcheck("all four items use 500 customers / 6 branches / 30 products / 180 pairs","500 / 6 / 30",f"{len(cust)} / {len(br)} / {len(allp)}",len(cust)==500 and len(br)==6 and len(allp)==30)
XV=pd.DataFrame(xchecks); assert (XV.Status=="PASS").all()
full=pd.concat([VAL,XV.assign(Check_ID=range(len(VAL)+1,len(VAL)+len(XV)+1))],ignore_index=True)
full.to_csv(W7/"validation_report_week07.csv",index=False); print(len(full),"checks written (",len(VAL),"from the four notebooks +",len(XV),"cross-checks )")

PASS customer revenue (W7-2) equals product revenue (W7-3) | expected 22958987430 | actual 22958987430 / 22958987430
PASS customer orders (W7-2) equal branch orders (W7-3) | expected 18033 | actual 18033 / 18033
PASS orders a month per branch: W7-1 series means equal W7-3 table (6 of 6) | expected equal | actual 6 of 6
PASS W7-1 mean monthly revenue times 72 equals total revenue | expected 22958987430 | actual 22958987430
PASS product units (W7-1) equal class A units (W7-3) for the ten class A products | expected equal | actual equal
PASS W7-4 constant-lead policy A fails 95% fill for the same 72 pairs as the W7-3 at-risk list | expected 72 | actual 72 / 72; identical sets: True
PASS W7-4 covers the same 180 pairs as the W7-3 category table | expected 180 | actual 180 / 180
PASS all four items use 500 customers / 6 branches / 30 products / 180 pairs | expected 500 / 6 / 30 | actual 500 / 6 / 30
113 checks written ( 105 from the four notebooks + 8 cross-checks )


## Part 4. Findings in one place

In [5]:
R1=f1; g=R1.groupby("group")
print("W7-1  Level forecast is enough for",int((pd.read_csv(o1/"w71_readiness_verdicts.csv").verdict=="Level forecast is enough").sum()),"of 40 series; the trend challenger wins by the pre-set bar in",int(R1.challenger_wins.sum()),".")
print("      Baseline error (median % of series mean):",g.mae_pct_plain.median().round(1).to_dict(),"; interval coverage",round(float(pd.read_csv(o1/'w71_backtest_detail.csv').inside.mean())*100,1),"%.")
c2=pd.read_csv(o2/"w72_concentration.csv").set_index("measure"); q=pd.read_csv(o2/"w72_summary_quartiles.csv")
print("W7-2  Top 10% of customers hold",round(c2.loc['top 10% share','value']*100,1),"% of revenue; Gini",round(c2.loc['Gini of revenue','value'],3),"; no group differs after Holm adjustment; behaviour does not persist.")
s4=pd.read_csv(o4/"w74_policy_summary.csv"); v4=s4[s4.lead_time_case.str.startswith("lead time varies")].set_index("policy")
print("W7-4  Lead time varies 7 to 35 days: fill A",round(v4.iloc[0].fill_rate*100,1),"%, B 95%",round(v4.loc[[i for i in v4.index if i.startswith('B 95')][0]].fill_rate*100,1),"%, B-LT 95%",round(v4.loc[[i for i in v4.index if i.startswith('B-LT 95')][0]].fill_rate*100,1),"% (in-sample; sales scale assumed, Q10 unresolved).")
print("W7-3  Class A:",len(cls),"products,",round(cls.revenue_share_pct.sum(),1),"% of revenue;",int(cls.below_portfolio_margin.sum()),"below the portfolio margin;",len(pd.read_csv(o3/'w73_at_risk_stock_pairs.csv')),"at-risk pairs (sales scale assumed, Q10 unresolved).")

W7-1  Level forecast is enough for 40 of 40 series; the trend challenger wins by the pre-set bar in 0 .
      Baseline error (median % of series mean): {'branch': 12.9, 'product': 12.7, 'total': 5.9} ; interval coverage 96.1 %.
W7-2  Top 10% of customers hold 14.3 % of revenue; Gini 0.125 ; no group differs after Holm adjustment; behaviour does not persist.
W7-4  Lead time varies 7 to 35 days: fill A 93.6 %, B 95% 98.8 %, B-LT 95% 99.2 % (in-sample; sales scale assumed, Q10 unresolved).
W7-3  Class A: 10 products, 81.5 % of revenue; 4 below the portfolio margin; 72 at-risk pairs (sales scale assumed, Q10 unresolved).
